In [0]:
df = spark.table('dbacademy.labuser15637412_1782372873.btc_usd_ytd')

#displays the dataframe
display(df)

snapped_at,price,market_cap,total_volume
2013-04-28T00:00:00.000Z,135.3,1.50051759E9,0.0
2013-04-29T00:00:00.000Z,141.96,1.575032004E9,0.0
2013-04-30T00:00:00.000Z,135.3,1.501657493E9,0.0
2013-05-01T00:00:00.000Z,117.0,1.29895155E9,0.0
2013-05-02T00:00:00.000Z,103.43,1.148667722E9,0.0
2013-05-03T00:00:00.000Z,91.01,1.011066494E9,0.0
2013-05-04T00:00:00.000Z,111.25,1.236351844E9,0.0
2013-05-05T00:00:00.000Z,116.79,1.298377788E9,0.0
2013-05-06T00:00:00.000Z,118.33,1.315992304E9,0.0
2013-05-07T00:00:00.000Z,106.4,1.1837665E9,0.0


In [0]:
df = spark.table('dbacademy.labuser15637412_1782372873.btc_usd_ytd')

# Convert to pandas
pdf = df.toPandas()

# Change snapped_at to date

pdf['snapped_at'] = pdf['snapped_at'].dt.date

# Rename snapped_at
pdf.rename(columns={'snapped_at': 'date'}, inplace=True)
pdf.drop(columns=['total_volume'], inplace=True)

# add target column
pdf['target'] = pdf['price'].shift(-1)
pdf.dropna(inplace=True)

pdf=pdf.sort_values(by='date')

display(pdf)

date,price,market_cap,target
2013-04-28,135.3,1.50051759E9,141.96
2013-04-29,141.96,1.575032004E9,135.3
2013-04-30,135.3,1.501657493E9,117.0
2013-05-01,117.0,1.29895155E9,103.43
2013-05-02,103.43,1.148667722E9,91.01
2013-05-03,91.01,1.011066494E9,111.25
2013-05-04,111.25,1.236351844E9,116.79
2013-05-05,116.79,1.298377788E9,118.33
2013-05-06,118.33,1.315992304E9,106.4
2013-05-07,106.4,1.1837665E9,112.64


In [0]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# Features and Target
features = ['price', 'market_cap']
x = pdf[features]
y = pdf['target']

#Split data
x_train, x_test, y_train, y_test = train_test_split(
    x,y,
    shuffle = False,
    test_size=0.2
)

#Train model
model = LinearRegression()
model.fit(x_train, y_train)

#Predict
Predictions = model.predict(x_test)

In [0]:
import pandas as pd

x_test_reset = x_test.reset_index(drop=True)
y_test_reset = y_test.reset_index(drop=True)

date_col = pdf.loc[x_test.index, 'date'].reset_index(drop=True)

#Build results table

results = pd.DataFrame({
    'date': date_col,
    'price': x_test_reset['price'].round(2),
    'Predicted Price': Predictions.round(2)
})

#Difference
results['difference'] = (results['Predicted Price'] - results['price']).round(2)
results['% Error'] = ((results['difference']/results['price'])*100).round(2)
#Results 
display(results)


date,price,Predicted Price,difference,% Error
2023-11-07,35031.27,35021.08,-10.19,-0.03
2023-11-08,35436.54,35426.0,-10.54,-0.03
2023-11-09,35795.08,35784.25,-10.83,-0.03
2023-11-10,36768.42,36756.69,-11.73,-0.03
2023-11-11,37344.25,37332.13,-12.12,-0.03
2023-11-12,37122.72,37110.53,-12.19,-0.03
2023-11-13,37067.7,37055.8,-11.9,-0.03
2023-11-14,36549.16,36537.69,-11.47,-0.03
2023-11-15,35545.2,35534.61,-10.59,-0.03
2023-11-16,37903.66,37890.95,-12.71,-0.03
